In [ ]:
%load_ext autoreload
%autoreload 2

## Data and dependencies setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df = pd.read_excel("../data/Training-data.xlsx")
print(df.head())

## Data Investigation
This step will involve me analysing the characteristics and features of Training-data.xlsx and creating a comprehensive plan on the choices I will make down the line later.

### Irrelevant features
From Feature_description.xlsx, we can see that 'tcprtt' is an irrelevant feature. This is because this feature is just the sum of 'synack' and 'ackdat'.

In [ ]:
# Checking whether 'tcprtt' is the exact sum of 'synack' and 'ackdat'
print("tcprtt == synack + ackdat:", df['tcprtt'].equals(df['synack'] + df['ackdat']))
# Checking the distribution of 'tcprtt' and the sum of 'synack' and 'ackdat'ArithmeticError
tcprtt = df['tcprtt']
synack_ackdat = df['synack'] + df['ackdat']

print("tcprtt")
print("\tMean: {}, Max: {}, Min: {}, ".format(np.mean(tcprtt), max(tcprtt), min(tcprtt)))

print("synack + ackdat")
print("\tMean: {}, Max: {}, Min: {}, ".format(np.mean(synack_ackdat), max(synack_ackdat), min(synack_ackdat)))

fig, axes = plt.subplots(1, 2)
axes[0].plot(tcprtt)
axes[0].set_title("tcprtt")
axes[0].set_xlabel("row index")

axes[1].plot(synack_ackdat, color = 'orange')
axes[1].set_title("synack + ackdat")
axes[1].set_xlabel("row index")
plt.show()


### Missing Values
This step involves me finding rows and columns with missing values within the dataset, and decide on how to handle them.

In [ ]:
# Check for the total number of missing values
print("Number of empty values: ", df.isna().sum().sum())

# Manual inspection of the values
for attr in df:
    print(df[attr].value_counts())

#### The "service" attribute
I have found that ~47% of "service" values is '-' -- unused. With around half of the column containing no values, I am considering the removal of this column. However, I am unsure whether the column being empty influences the 'label' attribute, and thereby removing this attribute will affect the model's ability to classify the data accurately. For now, I will keep the attribute within the dataset.

In [ ]:
print("Proportion of 'service' entries that are 'unused':")
print("{:.4f}".format(len(df[df["service"] == '-']) / len(df["service"])))

display(df.groupby(df['service'].eq('-'))['label'].value_counts(normalize=True))

### Duplicated Values
Finding any rows are columns that are repeated within the dataset, and investigating whether it is a legitimate duplication or an erroneous one.

Running the code below tells us that there are 355 duplicated rows and 2 duplicated columns within the dataset. As the duplicated rows make up less than 0.3% of the dataset, I will leave them in for now. For the columns, an entire column duplicating is very unlikely, especially when there is 140,000+ entries, therefore I am considering this an error/mistake, and I will drop the column.

In [ ]:
duplicated_rows = df.duplicated(keep=False)
duplicated_cols = df.T.duplicated(keep=False)

In [ ]:
print("No. of duplicated rows:",duplicated_rows.sum())
print("No. of duplicated cols:",duplicated_cols.sum())

display(df[duplicated_rows])
display(df.T[duplicated_cols])

### Data Types
Investigating the data types of the attributes and ensuring that they can be converted into appropriate types for the classification model. The dataset has three categorical attributes: proto, service, state. For each, I will create a one-hot-encoder, which is a binary array that represents the categorical value as a numeric type.

It is crucial to keep in mind that this preprocessing step will run after the data splitting, as doing it beforehand will lead to data leakage, where the information from the validation set can be present within the one-hot-encoder arrays. I still don't know how, but I am assumming that it will affect the length of the array, showing all the different possible values that it can take, even though it does not exist within the training set.

In [ ]:
# Investigating how many unique values each categorical attribute has
for attr in df.select_dtypes(include=['string']):
    print(attr, len(df[attr].unique()))

In [ ]:
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

encoded_df = encoder.fit_transform(df.select_dtypes(include=['string']))

print(encoder.get_feature_names_out())

### Scaling and Standardisation
I will now investigate whether any of the numeric attributes need to be scaled or standardised. The first cell will display all of the statistical characteristics of the data. Running the code, we can see that the maximum value within the dataset is 6033838853.939805, and the minimum is 0. This means that scaling and standardisation will be required. Otherwise, certain values will significantly dominate other values.

In [ ]:
import math
max_val = -math.inf
min_val = math.inf
for i in df.select_dtypes(include=['number']):
    print(i)
    print("\tmean:{:.2f}".format(np.mean(df[i]))) # mean
    print("\tmin:{:.2f}".format(min(df[i]))) # mean
    print("\tmax:{:.2f}".format(max(df[i]))) # mean
    if np.max(df[i]) > max_val:
        max_val = np.max(df[i])
    elif np.min(df[i]) < min_val:
        min_val = np.min(df[i])
print("Max: {}, Min: {}".format(max_val, min_val))

#### Z-Score

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns
df_standardised = df.copy()
df_standardised[numeric_cols] = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std()


display(df_standardised[numeric_cols])

This code will now show the new standardised mean, min, max, and standard deviation.

In [ ]:
max_val = -math.inf
min_val = math.inf
for i in df_standardised.select_dtypes(include=['number']):
    print(i)
    print("\tmean:{:.2f}".format(np.mean(df_standardised[i]))) # mean
    print("\tmin:{:.2f}".format(min(df_standardised[i]))) # mean
    print("\tmax:{:.2f}".format(max(df_standardised[i]))) # mean
    print("\tstd:{:.2f}".format(df_standardised[i].std()))
    if np.max(df_standardised[i]) > max_val:
        max_val = np.max(df_standardised[i])
    elif np.min(df_standardised[i]) < min_val:
        min_val = np.min(df_standardised[i])
print("Max: {}, Min: {}".format(max_val, min_val))

### Feature/Attribute Selection

In [ ]:
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(24,24))
sns.heatmap(corr, cmap='coolwarm', annot=True, fmt='.1f')

In [108]:
# GEMINI GENERATED

upper_tri = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

corr_pairs = upper_tri.unstack().dropna()

sorted_pairs = corr_pairs.reindex(corr_pairs.abs().sort_values(ascending=False).index)

display(sorted_pairs[sorted_pairs >= 0.95])

ct_ftp_cmd        is_ftp_login      1.000000
dwin              swin              0.999965
sloss             sbytes            0.997310
dloss             dbytes            0.993889
ct_srv_dst        ct_srv_src        0.981141
                  ct_dst_src_ltm    0.974134
sloss             spkts             0.971314
ct_dst_src_ltm    ct_srv_src        0.969176
dloss             dpkts             0.968572
ct_src_dport_ltm  ct_dst_ltm        0.967416
synack            tcprtt            0.964176
sbytes            spkts             0.963762
ackdat            tcprtt            0.960973
dbytes            dpkts             0.955898
dtype: float64

From the cell above, we can see that these pairs have unusually high correlation:
- dwin vs swin
- sloss vs sbytes
- dloss vs dbytes
- ct_srv_dst vs ct_srv_src
- ct_srv_dst vs ct_dst_src_ltm  
- sloss vs spkts
- ct_dst_src_ltm vs ct_srv_src
- dloss vs dpkts
- ct_src_dport_ltm vs ct_dst_ltm
- synack vs tcprtt
- sbytes vs spkts
- ackdat vs tcprtt
- dbytes vs dpkts

In [ ]:
# Checking dwin vs swin
print("dwin vs swin")
df[["swin", "dwin"]].describe()

print("sloss vs sbytes")
df[["swin", "dwin"]].describe()

dwin vs swin


,swin,dwin
count,145854.000000,145854.000000
mean,114.238965,114.234125
std,126.808932,126.808003
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,255.000000,255.000000
max,255.000000,255.000000
